# Create Fondation Vaincre Alzheimer Awards

Builds the awards table for **Fondation Vaincre Alzheimer** (funder_id
`4320315322`, IAMHRF expanded list, priority `299`) from vaincrealzheimer.org's
yearly funded-projects pages (2006-2026, 21 pages, 134 projects). PI/title/
institution/year 100%; EUR amounts on the 2006-2015 era pages only (45 rows —
recent pages publish no amounts; faithful nulls).

Source parquet: `s3://openalex-ingest/awards/vaincre_alzheimer/vaincre_alzheimer_grants.parquet`
(produced by `scripts/local/vaincre_alzheimer_to_s3.py`).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.vaincre_alzheimer_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/vaincre_alzheimer/vaincre_alzheimer_grants.parquet`;

In [ ]:
%sql
-- Check row count (should be ~134)
SELECT COUNT(*) as total_projects FROM openalex.awards.vaincre_alzheimer_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.vaincre_alzheimer_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.vaincre_alzheimer_raw LIMIT 5;

## Step 2: Create Vaincre Alzheimer Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.vaincre_alzheimer_awards
USING delta
AS
WITH
fva_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320315322  -- Fondation Vaincre Alzheimer
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        CASE WHEN g.amount IS NOT NULL THEN 'EUR' END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        CAST(NULL AS STRING) as funder_scheme,
        'vaincre_alzheimer' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'France' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.vaincre_alzheimer_raw g
    CROSS JOIN fva_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'vaincre_alzheimer' AND priority = 299;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    299 as priority
FROM openalex.awards.vaincre_alzheimer_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.vaincre_alzheimer_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_year
FROM openalex.awards.vaincre_alzheimer_awards LIMIT 10;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.vaincre_alzheimer_awards GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(amount) as with_amount
FROM openalex.awards.vaincre_alzheimer_awards GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator) as has_pi,
    COUNT(start_year) as has_year,
    ROUND(SUM(amount)/1000000, 2) as total_funding_million_eur
FROM openalex.awards.vaincre_alzheimer_awards;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as cnt
FROM openalex.awards.vaincre_alzheimer_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 15;

In [ ]:
%sql
SELECT COUNT(*) as fva_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'vaincre_alzheimer' AND priority = 299;